In [19]:
import numpy as np 
import inspect 
import logging 
np.set_printoptions( precision=5, suppress=True, linewidth=160, ) 
logging.getLogger().setLevel(logging.INFO)
import sys
import h5py

In [6]:
sys.path.insert(0, "/home/nv.krishnendu/work/3G/ce_stm/repo_2/CE_STM_CBC/population_fisher_seminumeric/")


In [7]:
print("\n" + "=" * 80) 
print("1. IMPORT / API CHECK") 
print("=" * 80) 
import pop_fisher 
import pop_fisher_term_I 
import pop_fisher_higher_order 
import pop_models 
print("pop_fisher imported OK") 
print("pop_fisher_term_I imported OK") 
print("pop_fisher_higher_order imported OK") 
print("pop_models imported OK")


1. IMPORT / API CHECK
pop_fisher imported OK
pop_fisher_term_I imported OK
pop_fisher_higher_order imported OK
pop_models imported OK


In [10]:
print("\nPublic API:") 
print(pop_fisher.__all__) 
print("\nMain wrapper:") 
print(inspect.signature(pop_fisher.compute_pop_fisher_full)) 
print("\nTerm-I function:") 
print(inspect.signature(pop_fisher.compute_pop_fisher)) 
print("\nCorrection function:") 
print(inspect.signature(pop_fisher.compute_correction_terms))


Public API:
['PopFisherResult', 'PopFisherFullResult', 'PopulationCatalogue', 'CorrectionTerms', 'load_population_catalogue', 'compute_pop_fisher', 'compute_pop_fisher_full', 'compute_correction_terms', 'compute_det_score']

Main wrapper:
(model, events, fim_phys=None, snr_threshold=None, params=None, fixed_parameters=None, n_total=None, n_det=None, h_lam_terms=0.03, h_theta=0.001, det_score=None, p_det_per_event=None, snr=None, sigma_rho=1.0, p_det_min=0.001, rcond=1e-12)

Term-I function:
(model, events, params=None, fixed_parameters=None, n_total=None, rcond=1e-12)

Correction function:
(model, events, fisher_source_frame, free_names, params_vec, phys_keys, snr=None, snr_threshold=None, det_score=None, p_det_per_event=None, sigma_rho=1.0, p_det_min=0.001, h_lam=0.03, h_theta=0.001)


In [14]:
print("compute_pop_fisher:") 
print(inspect.getsourcefile(pop_fisher.compute_pop_fisher)) 
print("\ncompute_correction_terms:") 
print(inspect.getsourcefile(pop_fisher.compute_correction_terms)) 
print("\nFree parameter wrapper:") 
print(inspect.getsource(pop_fisher._FreeParamWrapper))

compute_pop_fisher:
/home/nv.krishnendu/work/3G/ce_stm/repo_2/CE_STM_CBC/population_fisher_seminumeric/pop_fisher_term_I.py

compute_correction_terms:
/home/nv.krishnendu/work/3G/ce_stm/repo_2/CE_STM_CBC/population_fisher_seminumeric/pop_fisher_higher_order.py

Free parameter wrapper:
class _FreeParamWrapper:
    """Wraps a PopModel, fixing certain parameters and forwarding calls."""

    def __init__(self, model, free_names, fixed_parameters):
        self._model = model
        self.parameter_names = free_names
        self.fiducial = {name: model.fiducial[name] for name in free_names}
        self._fixed = fixed_parameters

        if hasattr(model, "analytic_score"):
            all_names = model.parameter_names
            free_idx = [all_names.index(name) for name in free_names]

            def analytic_score(
                events, params, _m=model, _fp=fixed_parameters, _idx=free_idx
            ):
                full_params = {**params, **_fp}
                all_scores = _

In [15]:
zmodel = pop_models.MadauDickinsonRedshift() 
print("Model:") 
print(zmodel) 
print("\nParameter names:") 
print(zmodel.parameter_names) 
print("\nFiducial:") 
print(zmodel.fiducial) 
print("\nPhysical event keys:") 
print(zmodel.phys_event_keys) 
print("\nanalytic_score source:") 
print(inspect.getsource(zmodel.analytic_score))

Model:
MadauDickinsonRedshift(alpha=2.7, beta=2.9, z_peak=1.9)

Parameter names:
['alpha', 'beta', 'z_peak']

Fiducial:
{'alpha': 2.7, 'beta': 2.9, 'z_peak': 1.9}

Physical event keys:
['redshift']

analytic_score source:
    def analytic_score(self, events, params):
        """
        Analytic score d ln p / d lambda for each event.
        (Look at Koustav's copy for the derivation.)

        Let u = (1+z)/(1+z_peak),  r = u^(alpha+beta).

        d ln p / d alpha  = ln(1+z) - r/(1+r) * ln(u)
        d ln p / d beta   =         - r/(1+r) * ln(u)
        d ln p / d z_peak = (alpha+beta) * r / ((1+r) * (1+z_peak))
        """
        redshift = events["redshift"]
        alpha = params["alpha"]
        beta = params["beta"]
        z_peak = params["z_peak"]

        ratio = (1.0 + redshift) / (1.0 + z_peak)
        r = ratio ** (alpha + beta)
        ln_ratio = numpy.log(ratio)
        frac = r / (1.0 + r)

        d_alpha = numpy.log1p(redshift) - frac * ln_ratio
        d_beta = -fr

In [17]:
file_path = "../../../3G/ce_stm/explore_data/cbc-stm/bbhs/networks/network_bbh_CE40km_1p0MW_aLIGO_coat_10.0hz_CE20km_1p0MW_aLIGO_coat_10.0hz_LIA+_10.0hz.h5"

In [20]:
with h5py.File(file_path, "r") as f:
    print("Top-level keys:")
    for key in f.keys():
        print("  ", key)

Top-level keys:
   condition_numbers
   covariance
   event_parameters
   fisher
   inversion_errors
   is_detected
   is_in_band
   sky_area_90
   snr


In [23]:
with h5py.File(file_path, "r") as f:
    print("snr:", f["snr"].shape, f["snr"].dtype)
    print("is_detected:", f["is_detected"].shape, f["is_detected"].dtype)
    print("is_in_band:", f["is_in_band"].shape, f["is_in_band"].dtype)

    print("\nFirst 10 SNRs:")
    print(f["snr"][:10])

    print("\nFirst 10 detected:")
    print(f["is_detected"][:10])

snr: (34166,) float64
is_detected: (34166,) bool
is_in_band: (34166, 3) bool

First 10 SNRs:
[  6.97536   7.70249  36.51515  13.55865  31.42727 142.37096  12.01882  13.25292   4.87725  51.36858]

First 10 detected:
[False False  True  True  True  True  True  True False  True]


In [24]:
with h5py.File(file_path, "r") as f:
    fisher = f["fisher"]

    print("shape:", fisher.shape)
    print("dtype:", fisher.dtype)

    print("\nFirst event Fisher:")
    print(fisher[0])

shape: (11, 11, 34166)
dtype: float64

First event Fisher:
[[ 1.17154e+02  1.58617e+01  6.39591e+06 ...  1.86093e+06  3.04015e+08  1.25838e+06]
 [-2.06650e+04 -4.08423e+03 -5.67867e+07 ... -3.37040e+07 -1.25658e+08 -1.42909e+06]
 [-1.57759e-03 -5.16950e-03 -1.15663e+01 ... -2.47439e+01 -2.81215e+02  3.79396e-01]
 ...
 [ 6.63287e+01  2.75889e+01  8.70222e+04 ...  5.08878e+04  2.88688e+05  4.68804e+03]
 [-9.75231e+02 -1.98325e+02 -3.37009e+06 ... -1.51961e+06 -3.94609e+07 -3.44597e+05]
 [-8.43025e+02 -1.34650e+02 -2.83284e+06 ... -1.38559e+06 -9.57468e+06 -1.52073e+05]]


In [27]:
import load


In [28]:

print("POPULATION_KEYS =", load.POPULATION_KEYS)

print(inspect.getsource(load.load_population_catalogue))

print(inspect.getsource(load.PopulationCatalogue))

POPULATION_KEYS = ['mass_1_source', 'mass_2_source', 'redshift']
def load_population_catalogue(
    file_path, snr_threshold, parameters=None, cosmology=None, with_fisher=True
):
    """
    ``FisherResults(file_path).load_population_catalogue(...)`` in one call.

    For scripts that do not otherwise need the ``FisherResults`` instance.
    See ``FisherResults.load_population_catalogue`` for the arguments.
    """
    return FisherResults(file_path).load_population_catalogue(
        snr_threshold,
        parameters=parameters,
        cosmology=cosmology,
        with_fisher=with_fisher,
    )

@dataclass
class PopulationCatalogue:
    """
    A detected catalogue in the form the population Fisher pipeline consumes.

    Attributes
    ----------
    events : dict
        Per-event source-frame column arrays keyed by bilby-style names
        ('mass_1_source', 'mass_2_source', 'redshift', and the derived
        'mass_ratio' = mass_2_source / mass_1_source).
    snr : ndarray
      

In [30]:
POPULATION_KEYS = [
    "mass_1_source",
    "mass_2_source",
    "redshift",
]

In [35]:
catalogue=load_population_catalogue(
    file_path,
    snr_threshold=30,
    parameters=None,
    cosmology=None,
    with_fisher=True,
)

2026-09-04 15:20:31,741 - INFO - Loaded Fisher file:
../../../3G/ce_stm/explore_data/cbc-stm/bbhs/networks/network_bbh_CE40km_1p0MW_aLIGO_coat_10.0hz_CE20km_1p0MW_aLIGO_coat_10.0hz_LIA+_10.0hz.h5
 which has detectors : CE40 (1.0MW aLIGO) CE20 (1.0MW), LIA+
 If I put an SNR threshold of: 10.0
 then I detect : 28696 / 34166 events
2026-09-04 15:20:34,279 - INFO - Rotated per-event Fisher into the source frame: (12268, 3, 3)
2026-09-04 15:20:34,280 - INFO - Loaded 12268 analysis events (is_detected & snr>=30) / 12268 above threshold / 34166 total injections from ../../../3G/ce_stm/explore_data/cbc-stm/bbhs/networks/network_bbh_CE40km_1p0MW_aLIGO_coat_10.0hz_CE20km_1p0MW_aLIGO_coat_10.0hz_LIA+_10.0hz.h5  [P_det = 0.3591]


In [36]:
print("\nfisher:")
print(catalogue.fisher.shape)
print(catalogue.fisher.dtype)

print("\ncounts:")
print("number_detected       =", catalogue.number_detected)
print("number_above_threshold=", catalogue.number_above_threshold)
print("number_total          =", catalogue.number_total)
print("P_det                 =", catalogue.p_det)


fisher:
(12268, 3, 3)
float64

counts:
number_detected       = 12268
number_above_threshold= 12268
number_total          = 34166
P_det                 = 0.3590704208862612


In [39]:
from pop_models import MadauDickinsonRedshift
from pop_fisher import compute_pop_fisher

model = MadauDickinsonRedshift()

result_I = compute_pop_fisher(
    model,
    catalogue.events,
    n_total=catalogue.number_total,
)

print("\n=== TERM I ===")
print("parameter names:", result_I.parameter_names)
print("Fisher shape:", result_I.fisher.shape)
print("n_events:", result_I.n_events)

print("\nFisher I:")
print(result_I.fisher)

2026-09-04 15:22:39,708 - INFO - Computing population Fisher: model=MadauDickinsonRedshift, free_params=['alpha', 'beta', 'z_peak'], fixed_parameters=[], N_events=12268
2026-09-04 15:22:39,711 - INFO - Mean score (= d ln P_det / d lambda): alpha=0.8353, beta=-0.01385, z_peak=0.586



=== TERM I ===
parameter names: ['alpha', 'beta', 'z_peak']
Fisher shape: (3, 3)
n_events: 12268

Fisher I:
[[ 703.65308 -141.65556 1278.01684]
 [-141.65556  188.64294 -599.46141]
 [1278.01684 -599.46141 3259.69047]]


In [42]:
score = result_I.score_matrix 
print("score shape =", score.shape) 
print("finite fraction =", np.isfinite(score).mean()) 
for i, name in enumerate(result_I.parameter_names): 
    s = score[:, i] 
    finite = np.isfinite(s) 
    print(f"\n{name}") 
    print(" finite =", finite.sum(), "/", len(s)) 
    print(" min =", np.nanmin(s)) 
    print(" max =", np.nanmax(s)) 
    print(" mean =", np.nanmean(s)) 
    print(" std =", np.nanstd(s)) 
    print("\nMean score:") 
    print(np.nanmean(score, axis=0))

score shape = (12268, 3)
finite fraction = 1.0

alpha
 finite = 12268 / 12268
 min = 0.03295395246758831
 max = 1.1144365426835159
 mean = 0.8353495088294101
 std = 0.23949277247176215

Mean score:
[ 0.83535 -0.01385  0.58604]

beta
 finite = 12268 / 12268
 min = -1.135574390842425
 max = 0.049725811069828714
 mean = -0.013847247053072826
 std = 0.12400334396306655

Mean score:
[ 0.83535 -0.01385  0.58604]

z_peak
 finite = 12268 / 12268
 min = 0.0058553929379042335
 max = 1.9277342362420486
 mean = 0.5860369009358561
 std = 0.515467511439421

Mean score:
[ 0.83535 -0.01385  0.58604]


In [44]:
FI = result_I.fisher 
sym_error = np.max(np.abs(FI - FI.T)) 
print("max |F - F.T| =", sym_error)

max |F - F.T| = 0.0


In [47]:
eig_I = np.linalg.eigvalsh((FI + FI.T) / 2) 
print("eigenvalues:") 
print(eig_I)

eigenvalues:
[  25.26498  237.67775 3889.04376]


In [48]:
F_events = catalogue.fisher

In [49]:
for j in range(min(5, len(F_events))): 
    F = F_events[j] 
    print(f"\nEvent {j}") 
    print(F) 
    sym = np.max(np.abs(F - F.T)) 
    eig = np.linalg.eigvalsh((F + F.T) / 2) 
    print(" symmetry error =", sym) 
    print(" eigenvalues =", eig)


Event 0
[[3.95807e+06 4.46759e+06 3.45481e+07]
 [4.46759e+06 5.04308e+06 3.89968e+07]
 [3.45481e+07 3.89968e+07 3.01561e+08]]
 symmetry error = 9.313225746154785e-10
 eigenvalues = [6.68232e+01 1.74874e+02 3.10562e+08]

Event 1
[[  170.54657   204.9885   2167.7012 ]
 [  204.9885    250.96087  2626.51724]
 [ 2167.7012   2626.51724 27762.72288]]
 symmetry error = 2.842170943040401e-14
 eigenvalues = [    1.2775      2.46445 28180.48837]

Event 2
[[5.20380e+07 5.67868e+07 6.41135e+08]
 [5.67868e+07 6.19698e+07 6.99647e+08]
 [6.41135e+08 6.99647e+08 7.89922e+09]]
 symmetry error = 7.450580596923828e-09
 eigenvalues = [3.36635e+02 1.30340e+03 8.01323e+09]

Event 3
[[   9189.91368   12672.5811   150866.1785 ]
 [  12672.5811    17478.24832  208063.1801 ]
 [ 150866.1785   208063.1801  2477980.00781]]
 symmetry error = 2.9103830456733704e-11
 eigenvalues = [      1.09766      11.80275 2504635.2694 ]

Event 4
[[2.63288e+06 2.20977e+07 8.10090e+07]
 [2.20977e+07 1.85782e+08 6.80514e+08]
 [8.1009

In [50]:
event_sym_errors = np.max( np.abs(F_events - np.swapaxes(F_events, 1, 2)), axis=(1, 2), )

In [51]:
print("\nPer-event Fisher symmetry:") 
print(" median error =", np.median(event_sym_errors)) 
print(" max error =", np.max(event_sym_errors))


Per-event Fisher symmetry:
 median error = 1.1641532182693481e-10
 max error = 0.000244140625


In [55]:

free_names = list(model.parameter_names) 
params_vec = np.array( [model.fiducial[name] for name in free_names], dtype=float, ) 
phys_keys = list(getattr(model, "phys_event_keys", [])) 
print("free_names =", free_names) 
print("phys_keys =", phys_keys) 
print("params_vec =", params_vec)

free_names = ['alpha', 'beta', 'z_peak']
phys_keys = ['redshift']
params_vec = [2.7 2.9 1.9]


In [53]:
finite_mask = np.all(np.isfinite(score), axis=1) 
print("\nFinite-score events:") 
print(finite_mask.sum(), "/", len(finite_mask))


Finite-score events:
12268 / 12268


In [58]:
from pop_fisher import ( compute_pop_fisher, compute_pop_fisher_full, compute_correction_terms, compute_det_score, )

In [61]:
corrections = compute_correction_terms( model, events_finite, fim_finite, free_names, params_vec, phys_keys, snr=snr_finite, snr_threshold=30, ) 
print("\nCorrection object:") 
print(corrections)

2026-09-04 15:29:26,745 - INFO - Computing det_score internally (snr_threshold=30, sigma_rho=1.0).
/home/nv.krishnendu/work/3G/ce_stm/repo_2/CE_STM_CBC/population_fisher_seminumeric/pop_fisher_higher_order.py:99: RuntimeWarning: overflow encountered in multiply
  dlnpdet_drho = 2.0 / (numpy.sqrt(2.0 * numpy.pi) * sigma_rho * erfcx(x_arg))
2026-09-04 15:29:27,148 - INFO - p_det range: [0.5000, 1.0000]; events with p_det < 0.99 (near threshold, non-zero gradient): 1115 / 12268. Term IV will be non-zero.
2026-09-04 15:29:27,148 - INFO - Term III weight 1/p_det(theta): min=1.0000, max=1.9998, mean=1.0221
2026-09-04 15:29:27,149 - INFO - Computing Terms II-V (correction terms)...



Correction object:
CorrectionTerms(fisher_II=array([[ 0.03256,  0.03284,  0.09175],
       [ 0.03284,  0.03293,  0.0921 ],
       [ 0.09175,  0.0921 , -3.00467]]), fisher_III=array([[ 0.03393,  0.0345 ,  0.10488],
       [ 0.0345 ,  0.03472,  0.10564],
       [ 0.10488,  0.10564, -3.18   ]]), fisher_IV=array([[ 0.01951,  0.02159,  1.46891],
       [ 0.02159,  0.02196,  1.47208],
       [ 1.46891,  1.47208, -3.08782]]), fisher_V=array([[-0.77499,  0.02097, -1.08781],
       [ 0.02097, -0.60442,  1.48131],
       [-1.08781,  1.48131, -1.80191]]), p_det=array([1.     , 0.92325, 1.     , ..., 1.     , 1.     , 1.     ]))


In [62]:
terms = { "II": corrections.fisher_II, "III": corrections.fisher_III, "IV": corrections.fisher_IV, "V": corrections.fisher_V, }

In [64]:
for name, F in terms.items(): 
    print("\n" + "-" * 60) 
    print(f"TERM {name}") 
    print("-" * 60) 
    print("shape =", F.shape) 
    print(np.array2string( F, precision=8, suppress_small=True, )) 
    print("\nSymmetry error:") 
    print(np.max(np.abs(F - F.T))) 
    eig = np.linalg.eigvalsh((F + F.T) / 2) 
    print("Eigenvalues:") 
    print(eig) 
    print("Diagonal:") 
    print(np.diag(F))


------------------------------------------------------------
TERM II
------------------------------------------------------------
shape = (3, 3)
[[ 0.03256209  0.03283714  0.0917499 ]
 [ 0.03283714  0.03292767  0.0920968 ]
 [ 0.0917499   0.0920968  -3.00466963]]

Symmetry error:
0.0
Eigenvalues:
[-3.01016 -0.00009  0.07108]
Diagonal:
[ 0.03256  0.03293 -3.00467]

------------------------------------------------------------
TERM III
------------------------------------------------------------
shape = (3, 3)
[[ 0.03393252  0.03450353  0.10487963]
 [ 0.03450353  0.0347215   0.10563559]
 [ 0.10487963  0.10563559 -3.17999717]]

Symmetry error:
0.0
Eigenvalues:
[-3.1868  -0.00018  0.07564]
Diagonal:
[ 0.03393  0.03472 -3.18   ]

------------------------------------------------------------
TERM IV
------------------------------------------------------------
shape = (3, 3)
[[ 0.01951271  0.0215943   1.46891229]
 [ 0.0215943   0.02196203  1.47207948]
 [ 1.46891229  1.47207948 -3.08781657]]

Sy

In [66]:
FII = corrections.fisher_II 
FIII = corrections.fisher_III 
FIV = corrections.fisher_IV 
FV = corrections.fisher_V 
Ftotal = FI + FII + FIII + FIV + FV 
print("\n" + "=" * 70) 
print("11. TOTAL FISHER") 
print("=" * 70) 
print(np.array2string( Ftotal, precision=8, suppress_small=True, )) 
print("\nCheck:") 
print("Ftotal - sum(terms) =") 
print(Ftotal - (FI + FII + FIII + FIV + FV)) 
sym_error = np.max(np.abs(Ftotal - Ftotal.T)) 
print("\nSymmetry error =", sym_error) 
eig_total = np.linalg.eigvalsh((Ftotal + Ftotal.T) / 2) 
print("Eigenvalues =", eig_total)


11. TOTAL FISHER
[[ 702.96409132 -141.54565275 1278.5945689 ]
 [-141.54565275  188.128129   -596.31028131]
 [1278.5945689  -596.31028131 3248.61607705]]

Check:
Ftotal - sum(terms) =
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Symmetry error = 0.0
Eigenvalues = [  25.01602  235.54271 3879.14956]


In [70]:
for i, name in enumerate(free_names): 
    total = Ftotal[i, i] 
    print(f"\n{name}") 
    if abs(total) < 1e-30: 
        print(" total approximately zero") 
        continue 
        for term_name, F in [ ("I", FI), ("II", FII), ("III", FIII), ("IV", FIV), ("V", FV), ]: 
            print( f" {term_name:>3s}: " f"{F[i,i]: .6e} " f"fraction={F[i,i]/total: .6e}" )


alpha

beta

z_peak


In [79]:
try: 
    cov = np.linalg.inv(Ftotal) 
    method = "ordinary inverse" 
except np.linalg.LinAlgError:
    cov = np.linalg.pinv(Ftotal, rcond=1e-12) 
    method = "pseudo-inverse" 
print("Method:", method) 
print("\nCovariance:") 
print(cov) 
sigma = np.sqrt(np.diag(cov)) 
print("\n1-sigma:") 
for name, s in zip(free_names, sigma): 
    print(f" {name:10s}: {s:.8g}") # Check F C = I 
    identity_check = Ftotal @ cov 
    print("\nFisher @ covariance:") 
    print(identity_check) 
    print("\nMaximum deviation from identity:") 
    print(np.max(np.abs(identity_check - np.eye(len(free_names)))))

Method: ordinary inverse

Covariance:
[[ 0.01118 -0.01324 -0.00683]
 [-0.01324  0.02839  0.01042]
 [-0.00683  0.01042  0.00491]]

1-sigma:
 alpha     : 0.10574085

Fisher @ covariance:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Maximum deviation from identity:
0.0
 beta      : 0.16848531

Fisher @ covariance:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Maximum deviation from identity:
0.0
 z_peak    : 0.07006606

Fisher @ covariance:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Maximum deviation from identity:
0.0


In [80]:
full = compute_pop_fisher_full( model, catalogue, ) 
print("Full Fisher:") 
print(full.fisher) 
print("\nDifference between manually assembled and full:") 
print(full.fisher - Ftotal) 
print("\nMaximum absolute difference:") 
print(np.max(np.abs(full.fisher - Ftotal)))

2026-09-04 15:39:45,756 - INFO - P_det(lambda) = 12268/34166 = 0.3591
2026-09-04 15:39:45,757 - INFO - Computing Term I (score Fisher)...
2026-09-04 15:39:45,757 - INFO - Computing population Fisher: model=MadauDickinsonRedshift, free_params=['alpha', 'beta', 'z_peak'], fixed_parameters=[], N_events=12268
2026-09-04 15:39:45,760 - INFO - Mean score (= d ln P_det / d lambda): alpha=0.8353, beta=-0.01385, z_peak=0.586
2026-09-04 15:39:45,761 - INFO - Events with finite scores: 12268
2026-09-04 15:39:45,902 - INFO - Computing det_score internally (snr_threshold=30.0, sigma_rho=1.0).
2026-09-04 15:39:46,368 - INFO - p_det range: [0.5000, 1.0000]; events with p_det < 0.99 (near threshold, non-zero gradient): 1115 / 12268. Term IV will be non-zero.
2026-09-04 15:39:46,368 - INFO - Term III weight 1/p_det(theta): min=1.0000, max=1.9998, mean=1.0221
2026-09-04 15:39:46,369 - INFO - Computing Terms II-V (correction terms)...
2026-09-04 15:40:00,411 - INFO - Full Fisher computed. Relative term s

Full Fisher:
[[ 702.96409 -141.54565 1278.59457]
 [-141.54565  188.12813 -596.31028]
 [1278.59457 -596.31028 3248.61608]]

Difference between manually assembled and full:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Maximum absolute difference:
4.547473508864641e-13
